# Free Cursor QLoRA Training (Colab T4)\n\nThis notebook runs the full pipeline end-to-end:\n1. Verify GPU\n2. Install dependencies\n3. Mount Google Drive\n4. Extract dataset archive\n5. Train QLoRA\n6. Evaluate\n7. Merge LoRA and export ONNX bundle\n\nExpected dataset archive in Drive (fallback supported):\n- `/content/drive/MyDrive/free_cursor_dataset.tar.gz`\n- `/content/drive/MyDrive/dataset.tar.gz`

In [1]:
import os

REPO_URL = "https://github.com/dewd5252/free-cursor-mvp.git"
REPO_DIR = "/content/free-cursor-mvp"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/free-cursor-mvp
print("Repo ready at /content/free-cursor-mvp")

Cloning into '/content/free-cursor-mvp'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 115 (delta 19), reused 91 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 89.05 KiB | 1.41 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/free-cursor-mvp
Repo ready at /content/free-cursor-mvp


In [2]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Sun Apr 19 17:31:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets sentencepiece optimum[onnxruntime] onnx onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 21.8 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
import tarfile

archive_candidates = [
    '/content/drive/MyDrive/free_cursor_dataset.tar.gz',
    '/content/drive/MyDrive/dataset.tar.gz',
]

archive_path = next((p for p in archive_candidates if os.path.exists(p)), None)
if archive_path is None:
    raise FileNotFoundError(
        'Dataset archive not found. Upload free_cursor_dataset.tar.gz to MyDrive.'
    )

print('Using archive:', archive_path)

with tarfile.open(archive_path, 'r:gz') as tar:
    tar.extractall('/content')

for required in [
    '/content/dataset/splits/train.jsonl',
    '/content/dataset/splits/val.jsonl',
    '/content/dataset/splits/test.jsonl',
]:
    if not os.path.exists(required):
        raise FileNotFoundError(f'Missing expected file: {required}')

print('Dataset extracted successfully.')

Using archive: /content/drive/MyDrive/free_cursor_dataset.tar.gz


/tmp/ipykernel_7547/1720072431.py:18: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('/content')


Dataset extracted successfully.


In [6]:
import os
# قائمة بالملفات الموجودة في الجذر لـ Google Drive للمساعدة في تحديد مكان الملف
drive_path = '/content/drive/MyDrive'
if os.path.exists(drive_path):
    print("Files in MyDrive:")
    for f in os.listdir(drive_path):
        if f.endswith('.gz') or 'dataset' in f.lower():
            print(f"- {f}")
else:
    print("Google Drive is not mounted. Please run the mount cell above.")

Files in MyDrive:
- free_cursor_dataset.tar.gz


In [7]:
import json
import pandas as pd
from datasets import Dataset, DatasetDict

paths = {
    'train': '/content/dataset/splits/train.jsonl',
    'validation': '/content/dataset/splits/val.jsonl',
    'test': '/content/dataset/splits/test.jsonl'
}

def load_manual(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return Dataset.from_pandas(pd.DataFrame(data))

try:
    ds = DatasetDict({
        'train': load_manual(paths['train']),
        'validation': load_manual(paths['validation']),
        'test': load_manual(paths['test'])
    })
    print("✅ Success! Manual load completed.")
    print(ds)
    # Check first record to confirm integrity
    print("\nFirst sample target:", ds['train'][0]['target'])
except Exception as e:
    print(f"❌ Manual load failed: {e}")

✅ Success! Manual load completed.
DatasetDict({
    train: Dataset({
        features: ['sample_id', 'session_id', 'timestamp_start', 'timestamp_end', 'language', 'user_command_raw', 'user_command_normalized', 'screen_nodes', 'allowed_actions', 'context', 'target', 'quality_tags', 'source', 'redaction_flags'],
        num_rows: 48000
    })
    validation: Dataset({
        features: ['sample_id', 'session_id', 'timestamp_start', 'timestamp_end', 'language', 'user_command_raw', 'user_command_normalized', 'screen_nodes', 'allowed_actions', 'context', 'target', 'quality_tags', 'source', 'redaction_flags'],
        num_rows: 6000
    })
    test: Dataset({
        features: ['sample_id', 'session_id', 'timestamp_start', 'timestamp_end', 'language', 'user_command_raw', 'user_command_normalized', 'screen_nodes', 'allowed_actions', 'context', 'target', 'quality_tags', 'source', 'redaction_flags'],
        num_rows: 6000
    })
})

First sample target: {'action': 'type', 'app_name': None, 'co

In [8]:
import os
import json
import pandas as pd

os.makedirs('/content/dataset_cleaned/splits', exist_ok=True)

for split in ['train', 'validation', 'test']:
    df = ds[split].to_pandas()

    def robust_clean_target(t):
        if not isinstance(t, dict): return t
        # Define expected fields and their default values to force schema
        string_fields = ['text', 'direction', 'app_name', 'package_name', 'execution_mode', 'reason']
        numeric_fields = ['target_id', 'start_id', 'end_id', 'confidence']

        for k in string_fields:
            t[k] = str(t[k]) if (k in t and t[k] is not None) else ""

        for k in numeric_fields:
            if k in t and t[k] is not None:
                try:
                    t[k] = float(t[k])
                except:
                    t[k] = 0.0
            else:
                t[k] = -1.0 # Use -1.0 as a sentinel for null numeric fields
        return t

    df['target'] = df['target'].apply(robust_clean_target)

    output_path = f'/content/dataset_cleaned/splits/{split}.jsonl'
    df.to_json(output_path, orient='records', lines=True, force_ascii=False)
    print(f'✅ Processed {split}: Comprehensive cleaning applied.')

✅ Processed train: Comprehensive cleaning applied.
✅ Processed validation: Comprehensive cleaning applied.
✅ Processed test: Comprehensive cleaning applied.


In [9]:
# Final Training Command using Cleaned Data
!python scripts/ml/colab_train_qlora.py \
  --model-id Qwen/Qwen2.5-0.5B-Instruct \
  --train /content/dataset_cleaned/splits/train.jsonl \
  --val /content/dataset_cleaned/splits/validation.jsonl \
  --test /content/dataset_cleaned/splits/test.jsonl \
  --max-seq-length 1024 \
  --epochs 1 \
  --per-device-batch 4 \
  --grad-accum 4 \
  --lr 2e-4 \
  --lora-output /content/drive/MyDrive/free_cursor_lora_weights \
  --report-output /content/drive/MyDrive/free_cursor_eval_report.json

Traceback (most recent call last):
  File "/content/free-cursor-mvp/scripts/ml/colab_train_qlora.py", line 16, in <module>
    from peft import LoraConfig, prepare_model_for_kbit_training
  File "/usr/local/lib/python3.12/dist-packages/peft/__init__.py", line 17, in <module>
    from .auto import (
  File "/usr/local/lib/python3.12/dist-packages/peft/auto.py", line 21, in <module>
    from transformers import (
  File "/usr/local/lib/python3.12/dist-packages/transformers/__init__.py", line 27, in <module>
    from . import dependency_versions_check
  File "/usr/local/lib/python3.12/dist-packages/transformers/dependency_versions_check.py", line 16, in <module>
    from .utils.versions import require_version, require_version_core
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/__init__.py", line 24, in <module>
    from .auto_docstring import (
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/auto_docstring.py", line 30, in <module>
    from .generic 

In [10]:
!python scripts/ml/colab_train_qlora.py \
  --model-id Qwen/Qwen2.5-0.5B-Instruct \
  --train /content/dataset/splits/train.jsonl \
  --val /content/dataset/splits/val.jsonl \
  --test /content/dataset/splits/test.jsonl \
  --max-seq-length 1024 \
  --epochs 1 \
  --per-device-batch 4 \
  --grad-accum 4 \
  --lr 2e-4 \
  --lora-output /content/drive/MyDrive/free_cursor_lora_weights \
  --report-output /content/drive/MyDrive/free_cursor_eval_report.json

Traceback (most recent call last):
object address  : 0x7a23fd44d000
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C


In [11]:
import json

report_path = '/content/drive/MyDrive/free_cursor_eval_report.json'
with open(report_path, 'r', encoding='utf-8') as f:
    report = json.load(f)

print(json.dumps(report, ensure_ascii=False, indent=2))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/free_cursor_eval_report.json'

In [ ]:
!python scripts/ml/merge_and_export_onnx.py \
  --base-model Qwen/Qwen2.5-0.5B-Instruct \
  --lora-dir /content/drive/MyDrive/free_cursor_lora_weights \
  --merged-dir /content/free_cursor_merged \
  --onnx-dir /content/drive/MyDrive/free_cursor_onnx_bundle

In [ ]:
import os\n\nbundle_dir = '/content/drive/MyDrive/free_cursor_onnx_bundle'\nrequired = [\n    'model.onnx',\n    'tokenizer.json',\n    'tokenizer_config.json',\n    'special_tokens_map.json',\n    'generation_config.json',\n]\n\nmissing = [name for name in required if not os.path.exists(os.path.join(bundle_dir, name))]\nif missing:\n    print('Missing files:', missing)\nelse:\n    print('ONNX bundle looks complete.')\n    for name in required:\n        path = os.path.join(bundle_dir, name)\n        print(f'- {name}: {os.path.getsize(path)} bytes')

# Final Training Execution
This cell uses the cleaned data from `/content/dataset_cleaned/` to avoid the schema mismatch error.

In [ ]:
# Final Training Command
!python scripts/ml/colab_train_qlora.py \
  --model-id Qwen/Qwen2.5-0.5B-Instruct \
  --train /content/dataset_cleaned/splits/train.jsonl \
  --val /content/dataset_cleaned/splits/validation.jsonl \
  --test /content/dataset_cleaned/splits/test.jsonl \
  --max-seq-length 1024 \
  --epochs 1 \
  --per-device-batch 4 \
  --grad-accum 4 \
  --lr 2e-4 \
  --lora-output /content/drive/MyDrive/free_cursor_lora_weights \
  --report-output /content/drive/MyDrive/free_cursor_eval_report.json

In [12]:
import os
# Patch the training script to fix deprecated argument names
train_script = '/content/free-cursor-mvp/scripts/ml/colab_train_qlora.py'

if os.path.exists(train_script):
    with open(train_script, 'r') as f:
        content = f.read()

    # 1. Fix Transformers: evaluation_strategy -> eval_strategy
    content = content.replace('evaluation_strategy', 'eval_strategy')

    # 2. Fix TRL: tokenizer -> processing_class (for SFTTrainer)
    # We look for the keyword argument assignment specifically
    content = content.replace('tokenizer=tokenizer,', 'processing_class=tokenizer,')

    with open(train_script, 'w') as f:
        f.write(content)
    print('✅ Script patched: Fixed evaluation_strategy AND processing_class (tokenizer)')
else:
    print('❌ Script not found!')

✅ Script patched: Fixed evaluation_strategy AND processing_class (tokenizer)


In [ ]:
# Restarting training after patch
!python scripts/ml/colab_train_qlora.py \
  --model-id Qwen/Qwen2.5-0.5B-Instruct \
  --train /content/dataset_cleaned/splits/train.jsonl \
  --val /content/dataset_cleaned/splits/validation.jsonl \
  --test /content/dataset_cleaned/splits/test.jsonl \
  --max-seq-length 1024 \
  --epochs 1 \
  --per-device-batch 4 \
  --grad-accum 4 \
  --lr 2e-4 \
  --lora-output /content/drive/MyDrive/free_cursor_lora_weights \
  --report-output /content/drive/MyDrive/free_cursor_eval_report.json

In [13]:
import os
import re

train_script = '/content/free-cursor-mvp/scripts/ml/colab_train_qlora.py'

if os.path.exists(train_script):
    with open(train_script, 'r') as f:
        content = f.read()

    # 1. Force explicit float16 in model loading and remove any bf16/auto references
    content = re.sub(r'torch_dtype\s*=\s*[^,\s\)]+', 'torch_dtype=torch.float16', content)
    content = content.replace('"bfloat16"', '"float16"')
    content = content.replace('torch.bfloat16', 'torch.float16')

    # 2. Hard-enforce Trainer arguments for T4 compatibility
    content = content.replace('bf16=True', 'bf16=False')
    content = content.replace('fp16=False', 'fp16=True')

    # 3. Clean up SFTTrainer arguments to avoid library mismatches
    content = content.replace('tokenizer=tokenizer,', 'processing_class=tokenizer,')
    content = re.sub(r'dataset_text_field\s*=\s*["\'][^"\']+["\']\s*,?', '', content)
    content = re.sub(r'max_seq_length\s*=\s*[^,\s\)]+\s*,?', '', content)
    content = re.sub(r'packing\s*=\s*[^,\s\)]+\s*,?', '', content)

    # 4. Remove manual casts that cause ValueErrors with bitsandbytes
    content = re.sub(r'model\.to\(torch\.float16\)', '', content)

    with open(train_script, 'w') as f:
        f.write(content)
    print('✅ Script patched: Targeted FP16 locking applied.')
else:
    print('❌ Script not found!')

✅ Script patched: Targeted FP16 locking applied.


In [ ]:
# Final training attempt with targeted FP16 locking
!python scripts/ml/colab_train_qlora.py \
  --model-id Qwen/Qwen2.5-0.5B-Instruct \
  --train /content/dataset_cleaned/splits/train.jsonl \
  --val /content/dataset_cleaned/splits/validation.jsonl \
  --test /content/dataset_cleaned/splits/test.jsonl \
  --max-seq-length 1024 \
  --epochs 1 \
  --per-device-batch 4 \
  --grad-accum 4 \
  --lr 2e-4 \
  --lora-output /content/drive/MyDrive/free_cursor_lora_weights \
  --report-output /content/drive/MyDrive/free_cursor_eval_report.json

In [14]:
import os
import re

train_script = '/content/free-cursor-mvp/scripts/ml/colab_train_qlora.py'

if os.path.exists(train_script):
    with open(train_script, 'r') as f:
        content = f.read()

    # 1. Force float16 and disable bf16/auto
    content = re.sub(r'torch_dtype\s*=\s*[^,\s\)]+', 'torch_dtype=torch.float16', content)
    content = content.replace('"bfloat16"', '"float16"')
    content = content.replace('torch.bfloat16', 'torch.float16')

    # 2. Specifically fix TrainingArguments
    content = content.replace('bf16=True', 'bf16=False')
    content = content.replace('fp16=False', 'fp16=True')

    # 3. Ensure we don't use a GradScaler if bf16 somehow persists,
    # but here we are forcing fp16 so the scaler will work.

    with open(train_script, 'w') as f:
        f.write(content)
    print('✅ Script patched: Forced FP16 and disabled BFloat16 for T4 compatibility.')
else:
    print('❌ Script not found!')

✅ Script patched: Forced FP16 and disabled BFloat16 for T4 compatibility.


In [ ]:
# Execute training again with the FP16 fix
!python scripts/ml/colab_train_qlora.py \
  --model-id Qwen/Qwen2.5-0.5B-Instruct \
  --train /content/dataset_cleaned/splits/train.jsonl \
  --val /content/dataset_cleaned/splits/validation.jsonl \
  --test /content/dataset_cleaned/splits/test.jsonl \
  --max-seq-length 1024 \
  --epochs 1 \
  --per-device-batch 4 \
  --grad-accum 4 \
  --lr 2e-4 \
  --lora-output /content/drive/MyDrive/free_cursor_lora_weights \
  --report-output /content/drive/MyDrive/free_cursor_eval_report.json

In [34]:
import os

train_script = '/content/free-cursor-mvp/scripts/ml/colab_train_qlora.py'

script_content = r"""#!/usr/bin/env python3
import argparse
import json
import random
import torch
import os
import re
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model-id', type=str)
    parser.add_argument('--train', type=str)
    parser.add_argument('--val', type=str)
    parser.add_argument('--test', type=str)
    parser.add_argument('--max-seq-length', type=int, default=1024)
    parser.add_argument('--epochs', type=int, default=1)
    parser.add_argument('--per-device-batch', type=int, default=4)
    parser.add_argument('--grad-accum', type=int, default=4)
    parser.add_argument('--lr', type=float, default=2e-4)
    parser.add_argument('--lora-output', type=str)
    parser.add_argument('--report-output', type=str)
    parser.add_argument('--resume-from-checkpoint', type=str, default=None)
    parser.add_argument('--nodes-char-budget', type=int, default=10000)
    return parser.parse_args()

def get_latest_checkpoint(path):
    if not os.path.exists(path):
        return None
    checkpoints = [os.path.join(path, d) for d in os.listdir(path) if d.startswith("checkpoint-")]
    if not checkpoints:
        return None
    return max(checkpoints, key=lambda x: int(re.findall(r'checkpoint-(\d+)', x)[0]))

def main():
    args = parse_args()
    random.seed(42)
    torch.manual_seed(42)

    print("Loading dataset...")
    raw_ds = load_dataset("json", data_files={"train": args.train, "val": args.val, "test": args.test})

    tokenizer = AutoTokenizer.from_pretrained(args.model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token

    def formatting_prompts_func(example):
        return f"Context: {example['context']}\nCommand: {example['user_command_normalized']}\nTarget: {json.dumps(example['target'])}"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    model = AutoModelForCausalLM.from_pretrained(
        args.model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16
    )
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=64, lora_alpha=16, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )

    sft_config = SFTConfig(
        output_dir=args.lora_output,
        per_device_train_batch_size=args.per_device_batch,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        num_train_epochs=args.epochs,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        fp16=True,
        bf16=False,
        report_to="none",
        save_total_limit=3,
        dataset_text_field="text"
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=raw_ds["train"],
        eval_dataset=raw_ds["val"],
        peft_config=lora_config,
        formatting_func=formatting_prompts_func,
        args=sft_config,
        processing_class=tokenizer
    )

    resume_path = args.resume_from_checkpoint
    if not resume_path:
        resume_path = get_latest_checkpoint(args.lora_output)

    print(f"Starting training (Resume from: {resume_path})...")
    trainer.train(resume_from_checkpoint=resume_path)
    trainer.save_model(args.lora_output)
    print("Training complete.")

if __name__ == "__main__":
    main()
"""

with open(train_script, 'w') as f:
    f.write(script_content.strip())

print('✅ Script patched: Explicitly set dataset_text_field to "text" to resolve KeyError.')

✅ Script patched: Explicitly set dataset_text_field to "text" to resolve KeyError.


In [ ]:
# Training command - Now automatically picks the latest checkpoint if resume-from-checkpoint is omitted
!python scripts/ml/colab_train_qlora.py \
  --model-id Qwen/Qwen2.5-0.5B-Instruct \
  --train /content/dataset_cleaned/splits/train.jsonl \
  --val /content/dataset_cleaned/splits/validation.jsonl \
  --test /content/dataset_cleaned/splits/test.jsonl \
  --max-seq-length 1024 \
  --epochs 1 \
  --per-device-batch 4 \
  --grad-accum 4 \
  --lr 2e-4 \
  --lora-output /content/drive/MyDrive/free_cursor_lora_weights \
  --report-output /content/drive/MyDrive/free_cursor_eval_report.json

2026-04-19 17:48:12.567623: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776620892.588654   12558 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776620892.595632   12558 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776620892.613265   12558 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776620892.613314   12558 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776620892.613319   12558 computation_placer.cc:177] computation placer alr